# SteelFlow: data inventory and relationship EDA

This notebook profiles the Parquet warehouse, confirms table grain, and documents the source-to-outcome flow before modeling.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


In [ ]:
files = sorted(DATA.glob('*.parquet'))
summary = []
for path in files:
    q = f"SELECT count(*) AS rows FROM read_parquet('{path.as_posix()}')"
    rows = int(con.execute(q).fetchone()[0])
    cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path.as_posix()}')").fetchdf()
    summary.append({'table': path.stem, 'rows': rows, 'columns': len(cols)})
catalog = pd.DataFrame(summary).sort_values('rows', ascending=False)
catalog.head(20)

In [ ]:
catalog.describe(include='all')

In [ ]:
# Inspect key entities and event tables
key_tables = ['customers', 'products', 'factories', 'warehouses', 'customer_orders', 'customer_order_lines', 'outbound_shipments', 'delivery_outcomes', 'production_batches']
for name in key_tables:
    path = DATA / f'{name}.parquet'
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path.as_posix()}')").fetchdf()
    print(f'\n{name}: {len(schema)} columns')
    display(schema[['column_name', 'column_type']])

In [ ]:
# Row-count scale
plt.figure(figsize=(12, 7))
sns.barplot(data=catalog.head(20), y='table', x='rows', color='#2563eb')
plt.title('Largest SteelFlow tables')
plt.xlabel('Rows'); plt.ylabel('')
plt.show()

## Data flow

Master tables describe entities. Transaction and event tables record orders, production, inventory, shipments, and payments. Outcome tables define supervised targets. Feature marts expose prediction-time variables and must be audited for leakage before use.